# 無料GPU画像生成環境（Kaggle × Civitai 3大モデル比較 × オリヴィアLoRA）

このノートブックは、Civitaiで人気の3つのIllustrious系基礎モデルをKaggle上で直接ダウンロード・切り替えて、同じプロンプト・同じシード（20261108）で生成し、画風を比較するための環境です。

### 比較対象モデル
1. **AnimeMix v8**（※Civitaiトークンが必要）
2. **EstellaXL**（※Civitaiトークンが必要）
3. **Holy Mix**（※Hugging Face直リンクのため**トークン不要で即実行可能！**）

※Civitaiのモデル（1と2）をダウンロードするには、Civitaiの無料APIキーが必要です（Settings → API Keys で取得）。
※トークンなしですぐ試したい場合は **`SELECTED_MODEL = 3`**（Holy Mix）をお試しください！

In [ ]:
# 1. GPU環境の確認
!nvidia-smi

In [ ]:
# 2. 必要なライブラリのインストール（不要なtorchaoを削除して競合防止）
!pip uninstall -y torchao
!pip install -q "diffusers>=0.30.0" transformers accelerate peft safetensors invisible-watermark protobuf sentencepiece

In [ ]:
# 3. アップロードしたLoRAファイルの自動探索
import os
import glob

lora_candidates = glob.glob("/kaggle/input/**/*.safetensors", recursive=True)
print("=== 検出されたsafetensorsファイル ===")
for f in lora_candidates:
    print(f)

if lora_candidates:
    lora_path = lora_candidates[0]
    print(f"\n[OK] 使用するLoRA: {lora_path}")
else:
    lora_path = None
    print("\n[警告] /kaggle/input に safetensors が見つかりません。右メニューの『Add Input』からLoRAデータセットを追加してください。")

In [ ]:
# 4. 比較するモデルの選択とダウンロード
import os

# =====================================================================
# 試したいモデル番号を選んでください：
# 3: Holy Mix（トークン不要ですぐ試せます！おすすめ）
# 1: AnimeMix v8（Civitaiトークンが必要）
# 2: EstellaXL（Civitaiトークンが必要）
SELECTED_MODEL = 3

# Civitaiの無料APIキー（https://civitai.com/user/account の API Keys で取得可能）
# 1 や 2 を試す場合はここに貼り付けてください（3の場合は空のままでOK）
CIVITAI_TOKEN = ""
# =====================================================================

MODELS = {
    1: {
        "name": "AnimeMix_v8",
        "url": "https://civitai.com/api/download/models/2290816",
        "filename": "animemix_v8.safetensors"
    },
    2: {
        "name": "EstellaXL",
        "url": "https://civitai.com/api/download/models/1929823",
        "filename": "estellaxl.safetensors"
    },
    3: {
        "name": "HolyMix_v1",
        "url": "https://huggingface.co/hollowstrawberry/holymix/resolve/main/HolyMix_illustriousXL_v1.safetensors",
        "filename": "holymix_v1.safetensors"
    }
}

target = MODELS[SELECTED_MODEL]
model_path = f"/kaggle/working/{target['filename']}"

# ダウンロードURL構築
download_url = target["url"]
if CIVITAI_TOKEN and "civitai.com" in download_url:
    download_url += f"?token={CIVITAI_TOKEN}"

# 古い壊れたファイル（1GB未満）や別モデルをクリーンアップ
for m in MODELS.values():
    check_path = f"/kaggle/working/{m['filename']}"
    if os.path.exists(check_path):
        # 1GB未満ならエラーページなので削除
        if os.path.getsize(check_path) < 1024 * 1024 * 1024 or check_path != model_path:
            print(f"不要または不完全なファイルを削除: {check_path}")
            os.remove(check_path)

if not os.path.exists(model_path):
    print(f"=== モデルをダウンロード中: {target['name']} ===")
    print(f"URL: {download_url.split('?')[0]}")
    !curl -L -o {model_path} "{download_url}"

# サイズチェック
file_size = os.path.getsize(model_path)
size_gb = file_size / (1024 ** 3)
if file_size < 1024 * 1024 * 1024:
    print(f"\n[エラー] ダウンロードされたファイルが小さすぎます ({file_size} bytes)。")
    print("CivitaiのAPIキー（CIVITAI_TOKEN）が未設定のため認証エラーとなった可能性があります。")
    print("まずはトークン不要な『SELECTED_MODEL = 3』をお試しいただくか、Civitaiキーを入力してください。")
else:
    print(f"\n[完了] 正常にダウンロード完了: {model_path} ({size_gb:.2f} GB)")

In [ ]:
# 5. パイプラインのロードとLoRAの適用（VRAM最適化）
import gc
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

gc.collect()
torch.cuda.empty_cache()

print(f"モデルを読み込み中: {target['name']} ...")
pipe = StableDiffusionXLPipeline.from_single_file(
    model_path,
    torch_dtype=torch.float16,
    use_safetensors=True
)

pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

if lora_path and os.path.exists(lora_path):
    print(f"LoRA重みを読み込み中: {lora_path} ...")
    pipe.load_lora_weights(lora_path, adapter_name="olivia")
    pipe.set_adapters(["olivia"], adapter_weights=[1.0])
    print("[OK] LoRA（Olivia von Ecclesia）を強度 1.0 で適用しました！")
else:
    print("LoRAなしでベースモデルのみを使用します。")

In [ ]:
# 6. テスト生成の実行（同一プロンプト・同一シードで比較）
from IPython.display import display

prompt = (
    "masterpiece, best quality, very aesthetic, newest, 1girl, solo, "
    "upper body, looking at viewer, "
    "liv, long blonde wavy hair, green eyes, soft smile, closed mouth, "
    "red and black royal dress with gold trim, ornate black gauntlets on both arms, black chest plate, "
    "elegant royal room background, "
    "clean line art, smooth shading, 2D anime style, vibrant colors"
)

negative_prompt = (
    "worst quality, low quality, bad anatomy, bad hands, missing fingers, extra digits, "
    "teeth, open mouth, "
    "cropped, text, watermark, signature, blurry, 3d, realistic, photorealistic, nsfw"
)

cfg_scale = 5.5
steps = 28

print(f"画像を生成中（モデル: {target['name']}, clip_skip=2, cfg={cfg_scale}, seed=20261108）...")
generator = torch.Generator(device="cuda").manual_seed(20261108)

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=832,
    height=1216,
    guidance_scale=cfg_scale,
    num_inference_steps=steps,
    clip_skip=2,
    generator=generator
).images[0]

output_dir = "/kaggle/working/outputs"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, f"olivia_compare_{target['name']}.png")
image.save(output_file)
print(f"\n[完了] 画像を保存しました: {output_file}")

display(image)